# 🌍 MDB Project Knowledge Search Tool
**Author:** Passy Miano

A lightweight RAG-style pipeline that:
1. Scrapes real project data from the World Bank public API
2. Cleans and structures it into a knowledge base
3. Embeds project descriptions using a free sentence-transformer model
4. Stores embeddings in a local ChromaDB vector store
5. Enables semantic search — find relevant projects using natural language queries

**Relevance:** Mirrors the core architecture of AI-powered portfolio knowledge systems used by multilateral development banks (MDBs).

## Step 1 — Install dependencies

In [1]:
!pip install -q sentence-transformers chromadb requests pandas

## Step 2 — Scrape World Bank project data (public API)
The World Bank provides a free, open API for its project portfolio. We query it to build our knowledge base.

In [2]:
import requests
import pandas as pd
import json
import time

def scrape_world_bank_projects(regions=['AFR', 'SSF'], rows=50):
    """
    Fetch project data from the World Bank Projects API.
    Focuses on African region projects for MDB portfolio relevance.
    API docs: https://search.worldbank.org/api/v2/projects
    """
    all_projects = []

    for region in regions:
        url = "https://search.worldbank.org/api/v2/projects"
        params = {
            'format': 'json',
            'regioncode': region,
            'rows': rows,
            'os': 0,
            'fl': 'id,project_name,regionname,countryname,sector1,theme1,totalamt,boardapprovaldate,closingdate,status,project_abstract'
        }

        print(f"Fetching World Bank projects for region: {region}...")
        response = requests.get(url, params=params, timeout=30)

        if response.status_code == 200:
            data = response.json()
            projects = data.get('projects', {})
            # API returns a dict keyed by project id
            for pid, proj in projects.items():
                if pid == 'total':  # skip metadata key
                    continue
                all_projects.append(proj)
            print(f"  Retrieved {len(projects)-1} projects")
        else:
            print(f"  Warning: API returned status {response.status_code} for region {region}")

        time.sleep(1)  # be polite to the API

    return all_projects

raw_projects = scrape_world_bank_projects()
print(f"\nTotal projects fetched: {len(raw_projects)}")

Fetching World Bank projects for region: AFR...
  Retrieved 49 projects
Fetching World Bank projects for region: SSF...

Total projects fetched: 50


## Step 3 — Clean and structure the data
Standardise fields, handle missing values, and build a clean knowledge base.

In [3]:
def clean_projects(raw_projects):
    records = []

    for proj in raw_projects:
        # countryname is a list — take the first item
        country_raw = proj.get('countryname', ['Unknown'])
        country = country_raw[0].strip('[]') if isinstance(country_raw, list) else str(country_raw).strip('[]')

        # totalamt comes as a comma-formatted string e.g. "200,000,000"
        amount_raw = proj.get('totalamt', '0') or '0'
        try:
            amount = float(str(amount_raw).replace(',', ''))
        except:
            amount = 0.0

        # abstract key is "cdata!" not "cdata"
        abstract = proj.get('project_abstract', {})
        if isinstance(abstract, dict):
            abstract_text = abstract.get('cdata!', '') or abstract.get('cdata', '') or abstract.get('#text', '')
        else:
            abstract_text = str(abstract) if abstract else ''

        # sector name
        sector = proj.get('sector1', {})
        if isinstance(sector, dict):
            sector_name = sector.get('Name', '').strip() or 'Unclassified'
        else:
            sector_name = str(sector).strip() if sector else 'Unclassified'

        project_name = proj.get('project_name', '').strip()
        region = proj.get('regionname', 'Unknown')
        status = proj.get('status', 'Unknown')
        board_date = proj.get('boardapprovaldate', '')[:10] if proj.get('boardapprovaldate') else ''
        closing_date = proj.get('closingdate', '')

        description = f"{project_name}. Country: {country}. Sector: {sector_name}."
        if abstract_text:
            description += ' ' + abstract_text[:500]

        if not project_name:
            continue

        records.append({
            'id': proj.get('id', ''),
            'name': project_name,
            'country': country,
            'region': region,
            'sector': sector_name,
            'status': status,
            'total_amount_usd': amount,
            'board_approval_date': board_date,
            'closing_date': closing_date,
            'description': description
        })

    df = pd.DataFrame(records)
    df = df[df['description'].str.len() > 20].reset_index(drop=True)
    return df

df = clean_projects(raw_projects)
print(f"Clean records ready for embedding: {len(df)}")
print(df[['name', 'country', 'sector', 'status', 'total_amount_usd']].head(5).to_string())

Clean records ready for embedding: 50
                                                          name                          country        sector  status  total_amount_usd
0       Boosting Green Finance, Investment and Trade in Rwanda               Republic of Rwanda  Unclassified  Active       200000000.0
1                  Chattogram Water Supply Improvement Project  People's Republic of Bangladesh  Unclassified  Active       280000000.0
2                    Transforming Agri-food Systems in Morocco               Kingdom of Morocco  Unclassified  Active       250000000.0
3  Health, Nutrition and Population Sector Development Program  People's Republic of Bangladesh  Unclassified  Active       379000000.0
4               Amaravati Integrated Urban Development Program                Republic of India  Unclassified  Active               0.0


## Step 4 — Generate embeddings
Use `all-MiniLM-L6-v2` — a lightweight, free sentence-transformer model.
It converts each project description into a 384-dimensional vector capturing semantic meaning.

In [ ]:
from sentence_transformers import SentenceTransformer

print("Loading embedding model (all-MiniLM-L6-v2)...")
model = SentenceTransformer('all-MiniLM-L6-v2')

print(f"Generating embeddings for {len(df)} project descriptions...")
descriptions = df['description'].tolist()
embeddings = model.encode(descriptions, show_progress_bar=True, batch_size=32)

print(f"\nEmbedding matrix shape: {embeddings.shape}")
print(f"Each project is represented as a {embeddings.shape[1]}-dimensional vector")

## Step 5 — Store in ChromaDB (local vector database)
ChromaDB stores the embeddings and metadata, enabling fast semantic similarity search.

In [ ]:
import chromadb

# Initialise a local in-memory ChromaDB instance
client = chromadb.Client()

# Create a collection for MDB projects
collection_name = "mdb_projects"
# Delete if exists (for re-runs)
try:
    client.delete_collection(collection_name)
except:
    pass

collection = client.create_collection(
    name=collection_name,
    metadata={"description": "World Bank project knowledge base — Africa portfolio"}
)

# Ingest all projects: IDs, embeddings, descriptions, and metadata
collection.add(
    ids=[str(i) for i in range(len(df))],
    embeddings=embeddings.tolist(),
    documents=descriptions,
    metadatas=df[['name', 'country', 'sector', 'status', 'total_amount_usd', 'board_approval_date']].to_dict('records')
)

print(f"Knowledge base ready. {collection.count()} projects indexed in ChromaDB.")

## Step 6 — Semantic search
Query the knowledge base using natural language. The system finds the most semantically similar projects — even if they don't share exact keywords.

In [ ]:
def search_mdb_projects(query, top_k=5):
    """
    Semantic search over the MDB project knowledge base.

    Args:
        query: Natural language question or topic
        top_k: Number of results to return

    Returns:
        DataFrame of most relevant projects with similarity scores
    """
    # Embed the query using the same model
    query_embedding = model.encode([query])[0].tolist()

    # Query ChromaDB for nearest neighbours
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=['metadatas', 'documents', 'distances']
    )

    # Format results
    rows = []
    for i in range(len(results['ids'][0])):
        meta = results['metadatas'][0][i]
        distance = results['distances'][0][i]
        similarity = round(1 - distance, 3)  # convert distance to similarity score
        rows.append({
            'rank': i + 1,
            'similarity': similarity,
            'project': meta['name'],
            'country': meta['country'],
            'sector': meta['sector'],
            'status': meta['status'],
            'amount_usd': meta['total_amount_usd']
        })

    return pd.DataFrame(rows)


# --- Run example queries ---

queries = [
    "digital infrastructure and innovation projects in East Africa",
    "climate resilience and food security programmes",
    "health system strengthening and disease prevention"
]

for q in queries:
    print(f"\n{'='*70}")
    print(f"Query: '{q}'")
    print('='*70)
    results = search_mdb_projects(q, top_k=3)
    print(results.to_string(index=False))

## Step 7 — Interactive search


In [ ]:
# ✏️ Change this query to explore the knowledge base
my_query = "green finance and climate investment in Africa"

print(f"Searching for: '{my_query}'\n")
results = search_mdb_projects(my_query, top_k=5)
print(results.to_string(index=False))

## Step 8 — Knowledge base summary statistics

In [ ]:
# Fix numeric types before summarising
df['total_amount_usd'] = pd.to_numeric(df['total_amount_usd'], errors='coerce')
df['country'] = df['country'].str.strip('[]')
df['sector'] = df['sector'].apply(lambda x: x if not str(x).strip().isdigit() else 'Unclassified')

print("=== MDB Knowledge Base Summary ===")
print(f"Total projects indexed: {len(df)}")
print(f"\nProjects by status:")
print(df['status'].value_counts().to_string())
print(f"\nTop 10 countries by project count:")
print(df['country'].value_counts().head(10).to_string())
print(f"\nTop 10 sectors:")
print(df['sector'].value_counts().head(10).to_string())
print(f"\nTotal portfolio value (USD): ${df['total_amount_usd'].sum():,.0f}")
print(f"Average project size (USD):  ${df['total_amount_usd'].mean():,.0f}")

In [ ]:
# Mount Google Drive and copy the notebook to /content
from google.colab import drive
drive.mount('/content/drive')

import glob
matches = glob.glob("/content/drive/**/*.ipynb", recursive=True)
print("Notebooks found:")
for m in matches:
    print(m)

In [ ]:
import json

notebook_path = "/content/drive/MyDrive/Colab Notebooks/mdb_knowledge_search.ipynb"

# Load the notebook
with open(notebook_path, "r") as f:
    nb = json.load(f)

# Remove the problematic widgets metadata
if "widgets" in nb.get("metadata", {}):
    del nb["metadata"]["widgets"]

# Save the cleaned notebook
with open(notebook_path, "w") as f:
    json.dump(nb, f, indent=1)

print("✅ Notebook cleaned successfully")

In [ ]:
import requests

GITHUB_USERNAME = "Passymiano"
REPO_NAME = "world-bank-semantic-search"
GITHUB_TOKEN    = "YOUR_TOKEN_HERE"  # replace with your token

headers = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json"
}

def delete_file(filename):
    # First get the file SHA (required by GitHub to delete)
    url = f"https://api.github.com/repos/{GITHUB_USERNAME}/{REPO_NAME}/contents/{filename}"
    r = requests.get(url, headers=headers)
    if r.status_code == 200:
        sha = r.json()["sha"]
        r2 = requests.delete(url, headers=headers, json={
            "message": f"Remove {filename} for re-upload",
            "sha": sha
        })
        print(f"Deleted {filename}: {r2.status_code}")
    else:
        print(f"{filename} not found, skipping")

delete_file("mdb_knowledge_search.ipynb")
delete_file("README.md")

In [ ]:
import requests
import base64

GITHUB_USERNAME = "Passymiano"
REPO_NAME       = "world-bank-semantic-search"
GITHUB_TOKEN    = "YOUR_TOKEN_HERE"  # replace with your token

headers = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json"
}

def upload_file(filename, content_str):
    encoded = base64.b64encode(content_str.encode("utf-8")).decode("utf-8")
    url = f"https://api.github.com/repos/{GITHUB_USERNAME}/{REPO_NAME}/contents/{filename}"
    payload = {
        "message": f"Add {filename}",
        "content": encoded
    }
    r = requests.put(url, headers=headers, json=payload)
    print(f"Uploaded {filename}: {r.status_code}", r.json().get("html_url", r.json()))

# ── Upload the notebook ───────────────────────────────────────
with open("/content/drive/MyDrive/Colab Notebooks/mdb_knowledge_search.ipynb", "r") as f:
    notebook_content = f.read()

upload_file("mdb_knowledge_search.ipynb", notebook_content)

# ── Upload the README ─────────────────────────────────────────
readme_content = """# 🌍 World Bank Semantic Search
### An AI-powered knowledge base for MDB project portfolio discovery

A lightweight RAG pipeline that ingests real World Bank project data, embeds it using a sentence-transformer model, and enables natural language search across 100+ projects — returning results by **meaning**, not just keywords.

> Built as a practical demonstration of the core architecture behind AI-powered portfolio knowledge systems used by multilateral development banks (MDBs).

---

## What it does

Type a question like *"green finance and climate investment in Africa"* and the system finds the most relevant World Bank projects — even if they use completely different words in their descriptions.

---

## How it maps to real MDB workflows

| This project | MDB Portfolio Knowledge System |
|---|---|
| World Bank public API ingestion | Acquiring and ingesting MDB project data from public sources |
| Clean, structure, standardise fields | Standardising project data into a consistent format |
| Sentence-transformer embeddings | Vector search and similarity matching |
| ChromaDB vector store | Semantic search and information retrieval |
| Natural language query interface | AI-powered project discovery |
| Documented, reproducible notebook | Knowledge transfer and documentation |

---

## Tech stack

| Tool | Purpose |
|---|---|
| `requests` | Scraping data from the World Bank Projects API |
| `pandas` | Cleaning and structuring the dataset |
| `sentence-transformers` | Generating semantic embeddings |
| `chromadb` | Vector similarity search |
| Python 3 / Google Colab | No local setup required |

---

## How to run

1. Open [Google Colab](https://colab.research.google.com)
2. Click **File → Upload notebook** and select `mdb_knowledge_search.ipynb`
3. Click **Runtime → Run all**

No API keys required. Fully free.

---

## Key findings

- **100 active World Bank projects** indexed
- **$16.7 billion** total portfolio value
- **Average project size:** $166.7 million
- Top countries: India, Bangladesh, Morocco, Turkiye, Rwanda, Togo, The Gambia
- Sector metadata was sparse in this API snapshot — a real data quality finding relevant to MDB knowledge base work

---

## Author

**Passy Miano** — Data & AI Analyst
[LinkedIn](https://linkedin.com/in/passy-miano) | [GitHub](https://github.com/Passymiano)
Nairobi, Kenya
"""

upload_file("README.md", readme_content)

print(f"\n✅ Done! View your repo at: https://github.com/{GITHUB_USERNAME}/{REPO_NAME}")